# 01 - Dataset and benchmark run
Run top to bottom in VS Code (select the `.venv` kernel). Everything is cached in `cache/benchmark.sqlite`, so re-running cells never repeats a successful API call.

In [ ]:
import sys, os
os.chdir(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
import pandas as pd, json
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 160)

## 1. Check keys and model IDs (one tiny call per model)
Edit `config/models.yaml` first if a model ID is not available to you.

In [ ]:
from src.main import main
import sys
sys.argv = ['router', 'verify-models']; main()

## 2. Build the benchmark
For the public half: fill `data/public_sources_manifest.yaml`, run `fetch-public`, then `build-data`. For a quick start use `--synthetic-only` (40 tasks; document this deviation).

In [ ]:
from src.data.build import build_data
# from src.data.fetch_public import fetch_public; fetch_public()   # after filling the manifest
tasks = build_data(synthetic_only=False)   # set True if you have no public sources yet

In [ ]:
df = pd.DataFrame(tasks)
df.groupby(['category', 'difficulty', 'source_type']).size().unstack(fill_value=0)

In [ ]:
df['input_words'] = df.input.str.split().str.len()
df.groupby('difficulty').input_words.describe()   # sanity check: difficulty should not just equal length

## 3. Run 60 tasks x 3 models (dry run first)

In [ ]:
from src.benchmark.run import run_benchmark
run_benchmark(dry_run=True)

In [ ]:
run_benchmark()   # resumable: stops after 5 consecutive errors (quota) - just run again later

In [ ]:
resp = pd.read_json('data/responses.jsonl', lines=True)
resp.groupby('tier').agg(ok=('status', lambda s: (s == 'ok').sum()), tokens_in=('input_tokens', 'sum'), tokens_out=('output_tokens', 'sum'), latency_ms=('latency_ms', 'median'))